### Hybrid Retrival Argumented Generation Evelution using RAGAS 

In [9]:
import warnings 
warnings.filterwarnings('ignore')

# Document load 
from langchain_community.document_loaders import PyPDFLoader 
loader  = PyPDFLoader('Static GK 2025.pdf')
pages = loader.load()

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
import hashlib

# Split Data 

spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=180)
text_spliter = spliter.split_documents(pages)
chunks = [i.page_content for i in text_spliter]
metadata = [i.metadata for i in text_spliter]
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]
print(f'print first 5 ids : {ids[:2]}')

print first 5 ids : ['df52eef7bfa55759b4642211e13e3020', '622d6c3b19974d6f39f9950848df1607']


In [11]:
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# client and collection create 
client = chromadb.PersistentClient(path="./Hybrid_RAG")
collection = client.get_or_create_collection(name="Hybrid_RAG",embedding_function=embedding_function)

if chunks:
    collection.add(
        ids=ids,
        documents=chunks , metadatas=metadata
    )
collection.count()

225

In [12]:
# LLM call 
import os 
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
load_dotenv()
try:
    key = os.getenv('GROQ_API_KEY')
    print(bool(key))
except Exception as e:
    print(str(e))
    
Groq = ChatGroq(model="qwen/qwen3.6-27b",max_tokens=2048)

test = Groq.invoke("hello llama?")
test.content

True


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "hello llama?"\n   - This is a greeting, but with a question mark, suggesting they might be checking if I\'m LLaMA or asking about my identity/capabilities.\n\n2.  **Identify Key Entities/Concepts:**\n   - "llama" likely refers to LLaMA (Large Language Model Meta AI), a family of open-weight LLMs developed by Meta.\n   - I need to clarify my identity: I am Qwen (通义千问), developed by Alibaba Group\'s Tongyi Lab, not LLaMA.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting politely\n   - Clarify my identity clearly and concisely\n   - Maintain a friendly, helpful tone\n   - Offer assistance\n\n4.  **Draft Response (Mental Refinement):**\n   Hello! I\'m actually Qwen, a large language model developed by Alibaba Group\'s Tongyi Lab, not LLaMA. How can I assist you today?\n\n5.  **Self-Correction/Verification:**\n   - Matches identity guidelines? Yes, clearly states I\'m Qwen.\n   - T

In [13]:
# Hybrid Corpus 
from rank_bm25 import BM25Okapi 
def tokenization(token):
    token = token.lower()
    token = token.split()
    return token 

tokens = [tokenization(i) for i in chunks]
bm_corpus = BM25Okapi(tokens)

print(f'sucussfully : {bm_corpus}')

sucussfully : <rank_bm25.BM25Okapi object at 0x128dab980>


In [14]:
def Hybrid_Retrive(query:str):
    query_re = Groq.invoke(f"write the query based on symentic search : {query}").content.strip()

    # Thats Vector DB retrival 
    result = collection.query(query_texts=[query_re] , n_results=5)
    dis  = result['distances'][0] 
    docs = result['documents'][0]
    threshold = 0.9
    print(f'the distance is : {dis}')
    dense_docs = []
    for i , d in zip(dis,docs):
        if threshold > i :
            dense_docs.append(d)
    # Thats Hybrid RAG Retrival using indexing 
    query_tokens = tokenization(query_re)
    score = bm_corpus.get_scores(query=query_tokens)
    def get_top_tokens (score , k=10):
        index = list(enumerate(score))
        idx_sorted = sorted(index,key=lambda x:x[1],reverse=True)
        return [doc for doc , _ in idx_sorted[:10]]
    index_tokens = [chunks[i] for i in get_top_tokens(score=score,k=10)]
    
    rrf_token = {}
    
    for rank , doc in enumerate(dense_docs):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(index_tokens):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
        
    marge = sorted(rrf_token.items() , key = lambda x:x[1] , reverse=True)
    get_docs = [i for i , _ in marge[:5]]
    
    return get_docs
    
def generation_answer(question:str , context_list:list):
    if not context_list :
        return "NOT Related Content"
    content_str = "\n\n".join(context_list) 
    
    prompt = f""" 
    Give answer based on the local document , if cant find out any related content 
    then direct type NOT related content 
    content : {content_str}
    question :{question}
    """
    response = Groq.invoke(prompt)
    
    return response.content

In [ ]:
# RAG evelute 
from datasets import Dataset 
from ragas import evaluate 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from langchain_community.embeddings import HuggingFaceEmbeddings 
user_input = []
retrival_context = []
response = []
reference = []

test_cases = [
    {
        "question": "who is first First Chief of Army Staff",
        "ground_truth": "General Maharaj Rajendra Singh Ji was the first Chief of Army Staff."
    },
    {
        "question": "Where is Malhargad Fort located?",
        "ground_truth": "Malhargad Fort is located in Sonori, near Saswad, Pune district, Maharashtra."
    },
    {
        "question": "Who built the Red Fort in Delhi?",
        "ground_truth": "Red Fort was built by Mughal Emperor Shah Jahan in 1648 AD."
    },
    {
        "question": "What is Purandar Fort famous for?",
        "ground_truth": "Purandar Fort is famous as the birthplace of Chhatrapati Sambhaji Maharaj."
    },
    {
        "question": "Which dynasty built Chitradurga Fort originally?",
        "ground_truth": "Chitradurga Fort was originally built by the Chalukyas between the 11th and 13th centuries."
    }
]
for item in test_cases:
    q =  item['question']
    truth = item['ground_truth']
    
    context = Hybrid_Retrive(query=q)
    LLM_answer = generation_answer(question=q , context_list=context)
    
    user_input.append(q)
    retrival_context.append(context)
    response.append(LLM_answer)
    reference.append(truth)
    
    data = {
        "user_input":user_input , 
        "retrieved_contexts":retrival_context , 
        "response":response , 
        "reference":reference
    }
    data = Dataset.from_dict(data)

    embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print(LLM_answer)

result = evaluate(
dataset= data, 
metrics=[Faithfulness(),AnswerRelevancy(),ContextPrecision(),ContextRecall()],
embeddings=embedding , 
llm=Groq,
raise_exceptions=False
)

df = result.to_pandas()
print(df)

the distance is : [0.6466941237449646, 0.6545263528823853, 0.6604306697845459, 0.6726630926132202, 0.6785009503364563]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.7077319622039795, 0.7169743776321411, 0.7239599227905273, 0.7358373403549194, 0.7451179027557373]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.5353990197181702, 0.542786717414856, 0.5707088708877563, 0.5813403129577637, 0.58970707654953]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

the distance is : [0.5792653560638428, 0.583594560623169, 0.5892015695571899, 0.6023147106170654, 0.6032506227493286]


In [1]:
!uv pip show ragas

Using Python 3.12.13 environment at: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv
Name: ragas
Version: 0.4.3
Location: /Users/debajyotihazra/Documents/MultiAgentic RAG /.venv/lib/python3.12/site-packages
Requires: appdirs, datasets, diskcache, instructor, langchain, langchain-community, langchain-core, langchain-openai, nest-asyncio, networkx, numpy, openai, pillow, pydantic, rich, scikit-network, tiktoken, tqdm, typer
Required-by:
